In [1]:
import numpy as np
import pandas as pd

from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u


# ============================================================
# FILES
# ============================================================

CEERS_FILE = "ceers_cat_v1.0.fits"

SPEC_FILE = (
    "hlsp_candels_hst_wfc3_egs_multi_v2_redshift-cat.txt"
)


# ============================================================
# 1. LOAD CEERS
# ============================================================

with fits.open(CEERS_FILE) as hdul:
    ceers_data = hdul[1].data

ceers = pd.DataFrame(
    {
        "CEERS_ID": ceers_data["NUMBER"],
        "RA": ceers_data["RA"],
        "DEC": ceers_data["DEC"],
        "LP_Z_MED": ceers_data["LP_Z_MED"],
        "LP_Z_BEST": ceers_data["LP_Z_BEST"],
    }
)

print("CEERS objects:", len(ceers))


# ============================================================
# 2. INSPECT KODRA FILE
# ============================================================

# The exact header/comment formatting may vary,
# so first inspect it.

with open(SPEC_FILE, "r") as f:
    for i in range(30):
        line = f.readline()
        print(line.rstrip())

CEERS objects: 87345
# ====================  EGS  ====================
# Catalog of z_best
# ==================================================
# 1 file
# 2 ID (CANDELS.EGS.F160W.v1b_1.photom.cat)
# 3 RA (CANDELS.EGS.F160W.v1b_1.photom.cat)
# 4 DEC (CANDELS.EGS.F160W.v1b_1.photom.cat)
# 5 z_best
# 6 z_best_type
# 7 z_spec
# 8 z_spec_ref
# 9 z_grism
# 10 mFDa4_z_peak
# 11 mFDa4_z_weight
# 12 mFDa4_z683_low
# 13 mFDa4_z683_high
# 14 mFDa4_z954_low
# 15 mFDa4_z954_high
# 16 HB4_z_peak
# 17 HB4_z_weight
# 18 HB4_z683_low
# 19 HB4_z683_high
# 20 HB4_z954_low
# 21 HB4_z954_high
# 22 Finkelstein_z_peak
# 23 Finkelstein_z_weight
# 24 Finkelstein_z683_low
# 25 Finkelstein_z683_high
# 26 Finkelstein_z954_low
# 27 Finkelstein_z954_high


In [2]:
spec = pd.read_csv(
    SPEC_FILE,
    sep=r"\s+",
    comment="#"
)

print(spec.columns.tolist())
print(spec.head())

['ALL_OPTIMIZED_PDFS_EGS_ID00001.pzd', '1', '215.3011028', '53.0512456', '1.947', 'p', '-1.0', '--', '-1.0.1', '1.95', '1.947.1', '1.524', '2.007', '1.447', '6.794', '1.95.1', '1.941', '1.799', '7.158', '0.45', '9.55', '1.95.2', '1.947.2', '1.524.1', '2.007.1', '1.447.1', '6.794.1', '6.84', '6.857', '6.793', '6.924', '6.728', '6.999', '2.2', '2.079', '1.764', '2.338', '1.497', '6.8', '7.13', '7.119', '7.053', '7.181', '6.972', '7.252', '1.76', '1.915', '1.683', '2.204', '1.536', '2.575', '1.94', '1.934', '1.879', '1.968', '1.492', '2.002']
   ALL_OPTIMIZED_PDFS_EGS_ID00001.pzd  1  215.3011028  53.0512456  1.947  p  \
0  ALL_OPTIMIZED_PDFS_EGS_ID00002.pzd  2   215.267148   53.027599  3.980  p   
1  ALL_OPTIMIZED_PDFS_EGS_ID00003.pzd  3   215.303229   53.052958  1.342  p   
2  ALL_OPTIMIZED_PDFS_EGS_ID00004.pzd  4   215.303816   53.053443  0.260  p   
3  ALL_OPTIMIZED_PDFS_EGS_ID00005.pzd  5   215.278725   53.035913  0.446  p   
4  ALL_OPTIMIZED_PDFS_EGS_ID00006.pzd  6   215.266135   53.

In [3]:
# ============================================================
# 3. IDENTIFY SPECTROSCOPIC OBJECTS
# ============================================================

print("\nAvailable columns:")
for col in spec.columns:
    print(col)

# Kodra catalog should contain something equivalent to:
# RA
# DEC
# z_spec
# z_spec_ref
# z_best_type

spec["z_spec"] = pd.to_numeric(
    spec["z_spec"],
    errors="coerce"
)

# Keep real spectroscopic measurements only.
#
# Adjust this if the catalog's missing-value convention
# is different after inspecting the file.

spec_clean = spec[
    np.isfinite(spec["z_spec"])
    & (spec["z_spec"] > 0)
].copy()

print("\nSpectroscopic redshifts available:")
print(len(spec_clean))

print("\nSpectroscopic z range:")
print(
    spec_clean["z_spec"].min(),
    spec_clean["z_spec"].max()
)


Available columns:
ALL_OPTIMIZED_PDFS_EGS_ID00001.pzd
1
215.3011028
53.0512456
1.947
p
-1.0
--
-1.0.1
1.95
1.947.1
1.524
2.007
1.447
6.794
1.95.1
1.941
1.799
7.158
0.45
9.55
1.95.2
1.947.2
1.524.1
2.007.1
1.447.1
6.794.1
6.84
6.857
6.793
6.924
6.728
6.999
2.2
2.079
1.764
2.338
1.497
6.8
7.13
7.119
7.053
7.181
6.972
7.252
1.76
1.915
1.683
2.204
1.536
2.575
1.94
1.934
1.879
1.968
1.492
2.002


KeyError: 'z_spec'

In [ ]:
# ============================================================
# 4. SKY COORDINATES
# ============================================================

ceers_coord = SkyCoord(
    ra=ceers["RA"].values * u.deg,
    dec=ceers["DEC"].values * u.deg
)

spec_coord = SkyCoord(
    ra=spec_clean["RA"].values * u.deg,
    dec=spec_clean["DEC"].values * u.deg
)


# ============================================================
# 5. MATCH NEAREST SPECTROSCOPIC SOURCE
# ============================================================

idx, sep2d, _ = ceers_coord.match_to_catalog_sky(spec_coord)

ceers["SPEC_INDEX"] = idx
ceers["SPEC_SEPARATION_ARCSEC"] = sep2d.arcsec


# ============================================================
# 6. REQUIRE CLOSE MATCH
# ============================================================

# Start conservatively at 0.3 arcsec.
#
# We can later test 0.2", 0.3", 0.5", etc.

MATCH_RADIUS = 0.3

matched = ceers[
    ceers["SPEC_SEPARATION_ARCSEC"] <= MATCH_RADIUS
].copy()

matched["Z_SPEC"] = spec_clean.iloc[
    matched["SPEC_INDEX"]
]["z_spec"].values


# Add source/reference if available

if "z_spec_ref" in spec_clean.columns:
    matched["Z_SPEC_REF"] = spec_clean.iloc[
        matched["SPEC_INDEX"]
    ]["z_spec_ref"].values


print("\n======================================")
print("MATCH RESULTS")
print("======================================")

print("CEERS sources:", len(ceers))
print("Spec catalog:", len(spec_clean))
print("Matched:", len(matched))

print(
    "Fraction of CEERS with spec-z:",
    len(matched) / len(ceers)
)


print("\nSeparation statistics:")
print(
    matched["SPEC_SEPARATION_ARCSEC"].describe()
)


print("\nSpectroscopic redshift statistics:")
print(
    matched["Z_SPEC"].describe()
)

In [6]:
import numpy as np
import pandas as pd

from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u


# ============================================================
# FILE PATHS
# ============================================================

CEERS_FILE = "ceers_cat_v1.0.fits"

SPEC_FILE = "hlsp_candels_hst_wfc3_egs_multi_v2_redshift-cat.txt"


# ============================================================
# HELPER: FITS BIG-ENDIAN -> NATIVE ENDIAN
# ============================================================

def native_array(arr):
    arr = np.asarray(arr)

    if arr.dtype.byteorder not in ("=", "|"):
        arr = arr.astype(arr.dtype.newbyteorder("="))

    return np.array(arr, copy=True)


# ============================================================
# 1. LOAD CANDELS EGS REDSHIFT CATALOG
# ============================================================

columns = [
    "file",
    "id",
    "ra",
    "dec",
    "z_best",
    "z_best_type",
    "z_spec",
    "z_spec_ref",
    "z_grism",

    "mfda4_z_peak",
    "mfda4_z_weight",
    "mfda4_z683_low",
    "mfda4_z683_high",
    "mfda4_z954_low",
    "mfda4_z954_high",

    "hb4_z_peak",
    "hb4_z_weight",
    "hb4_z683_low",
    "hb4_z683_high",
    "hb4_z954_low",
    "hb4_z954_high",

    "finkelstein_z_peak",
    "finkelstein_z_weight",
    "finkelstein_z683_low",
    "finkelstein_z683_high",
    "finkelstein_z954_low",
    "finkelstein_z954_high",

    "fontana_z_peak",
    "fontana_z_weight",
    "fontana_z683_low",
    "fontana_z683_high",
    "fontana_z954_low",
    "fontana_z954_high",

    "pforr_z_peak",
    "pforr_z_weight",
    "pforr_z683_low",
    "pforr_z683_high",
    "pforr_z954_low",
    "pforr_z954_high",

    "salvato_z_peak",
    "salvato_z_weight",
    "salvato_z683_low",
    "salvato_z683_high",
    "salvato_z954_low",
    "salvato_z954_high",

    "wiklind_z_peak",
    "wiklind_z_weight",
    "wiklind_z683_low",
    "wiklind_z683_high",
    "wiklind_z954_low",
    "wiklind_z954_high",

    "wuyts_z_peak",
    "wuyts_z_weight",
    "wuyts_z683_low",
    "wuyts_z683_high",
    "wuyts_z954_low",
    "wuyts_z954_high",
]

spec = pd.read_csv(
    SPEC_FILE,
    sep=r"\s+",
    comment="#",
    header=None,
    names=columns
)


# ============================================================
# 2. CONVERT IMPORTANT SPEC COLUMNS TO NUMERIC
# ============================================================

for col in [
    "ra",
    "dec",
    "z_best",
    "z_spec",
    "z_grism"
]:
    spec[col] = pd.to_numeric(
        spec[col],
        errors="coerce"
    )


# ============================================================
# 3. KEEP VALID SPECTROSCOPIC REDSHIFTS
# ============================================================

spec_clean = spec[
    np.isfinite(spec["ra"]) &
    np.isfinite(spec["dec"]) &
    np.isfinite(spec["z_spec"]) &
    (spec["z_spec"] > 0)
].copy()

spec_clean = spec_clean.reset_index(drop=True)


print("=" * 70)
print("CANDELS / SPECTROSCOPIC CATALOG")
print("=" * 70)

print("Total CANDELS rows:", len(spec))
print("Valid spectroscopic redshifts:", len(spec_clean))

print("\nz_best_type counts:")
print(
    spec["z_best_type"]
    .value_counts(dropna=False)
)

print("\nz_spec summary:")
print(
    spec_clean["z_spec"].describe()
)


# ============================================================
# 4. LOAD CEERS FITS
# ============================================================

bands = [
    "F115W",
    "F150W",
    "F200W",
    "F277W",
    "F356W",
    "F410M",
    "F444W",
    "F105W",
    "F125W",
    "F140W",
    "F160W",
    "F606W",
    "F814W",
    "F435W",
]

with fits.open(CEERS_FILE) as hdul:

    data = hdul[1].data

    ceers_dict = {
        "ceers_id": native_array(
            data["NUMBER"]
        ),

        "ra": native_array(
            data["RA"]
        ).astype(float),

        "dec": native_array(
            data["DEC"]
        ).astype(float),

        "z_lephare_med": native_array(
            data["LP_Z_MED"]
        ).astype(float),

        "z_lephare_best": native_array(
            data["LP_Z_BEST"]
        ).astype(float),
    }

    for band in bands:

        ceers_dict[f"{band}_flux"] = native_array(
            data[f"{band}_FLUX"]
        ).astype(float)

        ceers_dict[f"{band}_fluxerr"] = native_array(
            data[f"{band}_FLUXERR_EMP"]
        ).astype(float)


ceers = pd.DataFrame(ceers_dict)


print("\n" + "=" * 70)
print("CEERS CATALOG")
print("=" * 70)

print("CEERS objects:", len(ceers))
print("Columns loaded:", len(ceers.columns))


# ============================================================
# 5. CLEAN INVALID COORDINATES
# ============================================================

ceers = ceers[
    np.isfinite(ceers["ra"]) &
    np.isfinite(ceers["dec"])
].copy()

spec_clean = spec_clean[
    np.isfinite(spec_clean["ra"]) &
    np.isfinite(spec_clean["dec"])
].copy()

ceers = ceers.reset_index(drop=True)
spec_clean = spec_clean.reset_index(drop=True)


# ============================================================
# 6. CREATE SKY COORDINATES
# ============================================================

ceers_coords = SkyCoord(
    ra=ceers["ra"].to_numpy(dtype=float) * u.deg,
    dec=ceers["dec"].to_numpy(dtype=float) * u.deg
)

spec_coords = SkyCoord(
    ra=spec_clean["ra"].to_numpy(dtype=float) * u.deg,
    dec=spec_clean["dec"].to_numpy(dtype=float) * u.deg
)


# ============================================================
# 7. NEAREST-NEIGHBOUR CROSS-MATCH
# ============================================================

idx, sep2d, _ = ceers_coords.match_to_catalog_sky(
    spec_coords
)

idx = np.asarray(
    idx,
    dtype=np.int64
)

separations = np.asarray(
    sep2d.arcsec,
    dtype=np.float64
)

ceers["spec_index"] = idx
ceers["match_sep_arcsec"] = separations


# ============================================================
# 8. SHOW MATCH COUNTS AT DIFFERENT RADII
# ============================================================

print("\n" + "=" * 70)
print("MATCH COUNTS BY RADIUS")
print("=" * 70)

for radius in [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.75,
    1.00
]:

    count = np.sum(
        ceers[
            "match_sep_arcsec"
        ].to_numpy(dtype=float)
        <= radius
    )

    print(
        f'{radius:>4.2f}" : {count:>5} matches'
    )


# ============================================================
# 9. APPLY MAIN MATCH RADIUS
# ============================================================

MATCH_RADIUS_ARCSEC = 0.30

mask = (
    ceers[
        "match_sep_arcsec"
    ].to_numpy(dtype=float)
    <= MATCH_RADIUS_ARCSEC
)

matched = ceers.loc[mask].copy()

matched = matched.reset_index(drop=True)


# ============================================================
# 10. GET CORRESPONDING SPEC OBJECTS
# ============================================================

spec_indices = matched[
    "spec_index"
].to_numpy(dtype=np.int64)

matched_spec = (
    spec_clean
    .iloc[spec_indices]
    .reset_index(drop=True)
)


# ============================================================
# 11. ADD SPECTROSCOPIC INFORMATION
# ============================================================

matched["candels_id"] = (
    matched_spec["id"].to_numpy()
)

matched["spec_ra"] = (
    matched_spec["ra"]
    .to_numpy(dtype=float)
)

matched["spec_dec"] = (
    matched_spec["dec"]
    .to_numpy(dtype=float)
)

matched["z_spec"] = (
    matched_spec["z_spec"]
    .to_numpy(dtype=float)
)

matched["z_spec_ref"] = (
    matched_spec["z_spec_ref"]
    .to_numpy()
)

matched["z_best_type"] = (
    matched_spec["z_best_type"]
    .to_numpy()
)


# ============================================================
# 12. BASIC CROSS-MATCH RESULTS
# ============================================================

print("\n" + "=" * 70)
print("CROSS-MATCH RESULTS")
print("=" * 70)

print(
    "CEERS objects:",
    len(ceers)
)

print(
    "Objects with spectroscopic redshift:",
    len(spec_clean)
)

print(
    f'Matches within {MATCH_RADIUS_ARCSEC}" :',
    len(matched)
)

print(
    "Percentage of CEERS:",
    f"{100 * len(matched) / len(ceers):.3f}%"
)

print("\nMatch separation:")
print(
    matched[
        "match_sep_arcsec"
    ].describe()
)

print("\nSpectroscopic redshift:")
print(
    matched[
        "z_spec"
    ].describe()
)


# ============================================================
# 13. CHECK DUPLICATE SPEC MATCHES
# ============================================================

duplicates = matched[
    matched.duplicated(
        subset="spec_index",
        keep=False
    )
].sort_values(
    "spec_index"
)


print("\n" + "=" * 70)
print("DUPLICATE SPECTROSCOPIC MATCHES")
print("=" * 70)

print(
    "Rows involving duplicate matches:",
    len(duplicates)
)

print(
    "Unique spectroscopic objects duplicated:",
    duplicates[
        "spec_index"
    ].nunique()
)


if len(duplicates) > 0:

    print("\nExample duplicates:")

    print(
        duplicates[
            [
                "ceers_id",
                "candels_id",
                "spec_index",
                "z_spec",
                "match_sep_arcsec"
            ]
        ]
        .head(30)
        .to_string(index=False)
    )


# ============================================================
# 14. CREATE ONE-TO-ONE MATCHED SAMPLE
# ============================================================

matched_unique = (
    matched
    .sort_values(
        "match_sep_arcsec"
    )
    .drop_duplicates(
        subset="spec_index",
        keep="first"
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("ONE-TO-ONE MATCHES")
print("=" * 70)

print(
    "Original matches:",
    len(matched)
)

print(
    "Unique one-to-one matches:",
    len(matched_unique)
)

print(
    "Removed duplicate matches:",
    len(matched)
    -
    len(matched_unique)
)


# ============================================================
# 15. ASTROMETRIC OFFSET
# ============================================================

dec_rad = np.deg2rad(
    matched_unique[
        "dec"
    ].to_numpy(dtype=float)
)

delta_ra_arcsec = (
    matched_unique[
        "ra"
    ].to_numpy(dtype=float)
    -
    matched_unique[
        "spec_ra"
    ].to_numpy(dtype=float)
) * 3600 * np.cos(dec_rad)

delta_dec_arcsec = (
    matched_unique[
        "dec"
    ].to_numpy(dtype=float)
    -
    matched_unique[
        "spec_dec"
    ].to_numpy(dtype=float)
) * 3600


matched_unique[
    "delta_ra_arcsec"
] = delta_ra_arcsec

matched_unique[
    "delta_dec_arcsec"
] = delta_dec_arcsec


print("\n" + "=" * 70)
print("ASTROMETRIC OFFSET")
print("=" * 70)

print(
    "Median delta RA:",
    np.median(delta_ra_arcsec),
    "arcsec"
)

print(
    "Median delta DEC:",
    np.median(delta_dec_arcsec),
    "arcsec"
)

print(
    "Mean delta RA:",
    np.mean(delta_ra_arcsec),
    "arcsec"
)

print(
    "Mean delta DEC:",
    np.mean(delta_dec_arcsec),
    "arcsec"
)

print(
    "Std delta RA:",
    np.std(delta_ra_arcsec),
    "arcsec"
)

print(
    "Std delta DEC:",
    np.std(delta_dec_arcsec),
    "arcsec"
)


# ============================================================
# 16. CLEAN LEPHARE MISSING VALUES
# ============================================================

for col in [
    "z_lephare_med",
    "z_lephare_best"
]:

    matched_unique.loc[
        matched_unique[col] < 0,
        col
    ] = np.nan


# ============================================================
# 17. REDSHIFT DISTRIBUTION
# ============================================================

bins = [
    0,
    0.5,
    1,
    1.5,
    2,
    3,
    4,
    6,
    10,
    20
]

matched_unique[
    "z_bin"
] = pd.cut(
    matched_unique[
        "z_spec"
    ],
    bins=bins
)


print("\n" + "=" * 70)
print("SPECTROSCOPIC REDSHIFT DISTRIBUTION")
print("=" * 70)

print(
    matched_unique[
        "z_bin"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 18. LEPHARE VS SPECTROSCOPY
# ============================================================

comparison = matched_unique[
    np.isfinite(
        matched_unique[
            "z_lephare_med"
        ]
    )
    &
    np.isfinite(
        matched_unique[
            "z_spec"
        ]
    )
].copy()


comparison[
    "delta_z"
] = (
    comparison[
        "z_lephare_med"
    ]
    -
    comparison[
        "z_spec"
    ]
)


comparison[
    "delta_z_norm"
] = (
    comparison[
        "z_lephare_med"
    ]
    -
    comparison[
        "z_spec"
    ]
) / (
    1
    +
    comparison[
        "z_spec"
    ]
)


print("\n" + "=" * 70)
print("LEPHARE VS SPECTROSCOPY")
print("=" * 70)

print(
    "Number of comparable objects:",
    len(comparison)
)

print(
    "Mean absolute z error:",
    np.mean(
        np.abs(
            comparison[
                "delta_z"
            ]
        )
    )
)

print(
    "Median |dz/(1+z)|:",
    np.median(
        np.abs(
            comparison[
                "delta_z_norm"
            ]
        )
    )
)


# ============================================================
# 19. NMAD
# ============================================================

median_delta = np.median(
    comparison[
        "delta_z_norm"
    ]
)

sigma_nmad = (
    1.48
    *
    np.median(
        np.abs(
            comparison[
                "delta_z_norm"
            ]
            -
            median_delta
        )
    )
)

print(
    "sigma_NMAD:",
    sigma_nmad
)


# ============================================================
# 20. CATASTROPHIC OUTLIER FRACTION
# ============================================================

comparison[
    "catastrophic_outlier"
] = (
    np.abs(
        comparison[
            "delta_z_norm"
        ]
    )
    > 0.15
)

outlier_fraction = (
    comparison[
        "catastrophic_outlier"
    ].mean()
)


print(
    "Catastrophic outlier fraction:",
    outlier_fraction
)

print(
    "Catastrophic outlier percentage:",
    f"{outlier_fraction * 100:.2f}%"
)


# ============================================================
# 21. SHOW FIRST MATCHES
# ============================================================

show_columns = [
    "ceers_id",
    "candels_id",
    "ra",
    "dec",
    "spec_ra",
    "spec_dec",
    "z_spec",
    "z_lephare_med",
    "z_lephare_best",
    "z_spec_ref",
    "match_sep_arcsec",
    "delta_ra_arcsec",
    "delta_dec_arcsec",
]

print("\n" + "=" * 70)
print("FIRST 20 UNIQUE MATCHES")
print("=" * 70)

print(
    matched_unique[
        show_columns
    ]
    .head(20)
    .to_string(index=False)
)


# ============================================================
# 22. SAVE FILES
# ============================================================

matched.to_csv(
    "ceers_specz_all_matches.csv",
    index=False
)

matched_unique.to_csv(
    "ceers_specz_unique_matches.csv",
    index=False
)

comparison.to_csv(
    "ceers_specz_lephare_comparison.csv",
    index=False
)


print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

print(
    "Saved: ceers_specz_all_matches.csv"
)

print(
    "Saved: ceers_specz_unique_matches.csv"
)

print(
    "Saved: ceers_specz_lephare_comparison.csv"
)

CANDELS / SPECTROSCOPIC CATALOG
Total CANDELS rows: 41457
Valid spectroscopic redshifts: 2540

z_best_type counts:
z_best_type
p    38378
s     2540
g      539
Name: count, dtype: int64

z_spec summary:
count    2540.000000
mean        0.900252
std         0.572148
min         0.021500
25%         0.490861
50%         0.750726
75%         1.170550
max         3.638700
Name: z_spec, dtype: float64

CEERS CATALOG
CEERS objects: 87345
Columns loaded: 33

MATCH COUNTS BY RADIUS
0.10" :    20 matches
0.15" :    76 matches
0.20" :   540 matches
0.25" :  1108 matches
0.30" :  1204 matches
0.40" :  1294 matches
0.50" :  1367 matches
0.75" :  1590 matches
1.00" :  1932 matches

CROSS-MATCH RESULTS
CEERS objects: 87345
Objects with spectroscopic redshift: 2540
Matches within 0.3" : 1204
Percentage of CEERS: 1.378%

Match separation:
count    1204.000000
mean        0.202034
std         0.036487
min         0.051470
25%         0.182053
50%         0.203557
75%         0.223552
max         0.2999